# Poverty Map (Facebook Marketing)

Estimate audience (# people)

## Setup

### Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!ln -s /content/drive/MyDrive/Colab\ Notebooks/SES-Inference /mydrive

Mounted at /content/drive


### Facebook Marketing

In [ ]:
!pip install facebook_business
!pip install geopandas

### Dependencies

In [3]:
### System
import sys
from tqdm import tqdm
import pandas as pd

### Local
sys.path.append('/mydrive/libs/')
%set_env PYTHONPATH=/env/python:/mydrive/libs/

%load_ext autoreload
%autoreload 2

from facebook.marketing import FacebookMarketing
from utils import ios
from maps import geo

env: PYTHONPATH=/env/python:/mydrive/libs/


## Examples

In [5]:
### TOKENS
fn_tokens = '/mydrive/data/Facebook/MarketingAPI/tokens/tokens1.json'
tokens = ios.load_json(fn_tokens)
access_token = tokens['access_token']
app_secret = tokens['app_secret']
ad_account_id = tokens['ad_account_id']
app_id = tokens['app_id']

### CONNECT
fbm = FacebookMarketing(app_id=app_id,
                        app_secret=app_secret,
                        access_token=access_token,
                        ad_account_id=ad_account_id)
fbm.init()

### LOCATIONS (lon,lat)
points = []
# points.append([32.02786014765624,0.4653902099549784]) # Kampala (dark)
# points.append([32.15326611328126,0.37593062258811094]) # Kampala (grey)
# points.append([32.046836059570325,0.40373912169855747]) # Kampala (white)
# points.append([8.66414963290783,50.118508372433]) # Frankfurt (white)
# points.append([-73.97282574075544,40.779868876549926]) # Manhattan (white)
# points.append([33.4583298577, 3.2037499184900002]) # Kampala (wrong location?)
# points.append([16.349574526841728, 48.166388205677045]) # Vienna
points.append([6.974701178001343, 50.936673977734294]) # Cologne

### USER PROFILES
# profiles = {'os_android':FacebookMarketing.OS_ANDROID, 
#             'behavior_frequent_traveler':FacebookMarketing.BEHAVIOR_FREQUENT_TRAVELER, 
#             'edu_highschool':FacebookMarketing.EDU_HIGHSCHOOL}

# profiles = {'le_away':{'life_events':[{'id':6003053860372, 'name':'Away from hometown'},{'id':6003053857372, 'name':'Away from family'}]},
#             'le_away_from_hometown':{'life_events':[{'id':6003053860372, 'name':'Away from hometown'}]},
#             'le_away_from_family':{'life_events':[{'id':6003053857372, 'name':'Away from family'}]}}

profiles = {'beh_browser':{'behaviors':[{'id':6015547847583, 'name':'Facebook access (browser): Firefox'},
                                          {'id':6015547900583, 'name':'Facebook access (browser): Chrome'},
                                          {'id':6015593608983, 'name':'Facebook access (browser): Safari'},
                                          {'id':6015593652183, 'name':'Facebook access (browser): Opera'},
                                          {'id':6015593776783, 'name':'Facebook access (browser): Internet Explorer'}]},
            'beh_firefox':{'behaviors':[{'id':6015547847583, 'name':'Facebook access (browser): Firefox'}]},
            'beh_chrome':{'behaviors':[{'id':6015547900583, 'name':'Facebook access (browser): Chrome'}]},
            'beh_safari':{'behaviors':[{'id':6015593608983, 'name':'Facebook access (browser): Safari'}]},
            'beh_opera':{'behaviors':[{'id':6015593652183, 'name':'Facebook access (browser): Opera'}]},
            'beh_explorer':{'behaviors':[{'id':6015593776783, 'name':'Facebook access (browser): Internet Explorer'}]},
            }
### QUERY
df = pd.DataFrame()
radius=geo.MILE_TO_M / geo.KM_TO_M
unit=FacebookMarketing.UNIT_KM
for id, point in enumerate(points):
  df = df.append(pd.DataFrame(index=[id]))

  for k,v in profiles.items():
    result,_ = fbm.get_reach_estimate(lon=point[0], lat=point[1], radius=radius, unit=unit, specific_params=v)
    df.loc[id,k] = None if result is None else result['users']
    
df.head(10)

,beh_browser,beh_firefox,beh_chrome,beh_safari,beh_opera,beh_explorer
0,9800.0,1400.0,4800.0,3500.0,1000.0,1000.0


In [15]:
!rm -rf /mydrive/cache/FBM*

## All-in-one (fast)

### Survey data (clusters)

In [ ]:
### by DHS cluster
fn = '/mydrive/data/Uganda/results/DHS2016_MIS2018_iwi_cluster.csv'
df_dhs = ios.load_csv(fn, index_col=0)
df_dhs.head()

,DHSCC,DHSYEAR,DHSCLUST,URBAN_RURA,LATNUM,LONGNUM,SOURCE,ALT_GPS,ALT_DEM,DATUM,mean_iwi,SES
0,UG,2016,1,1,0.320188,32.568206,GPS,9999.0,1197.0,WGS84,43.300000,rich
1,UG,2016,2,1,0.340653,32.593627,GPS,9999.0,1179.0,WGS84,24.967857,rich
2,UG,2016,3,1,0.313103,32.566556,GPS,9999.0,1189.0,WGS84,34.732000,rich
3,UG,2016,4,1,0.353368,32.558144,GPS,9999.0,1181.0,WGS84,36.060714,rich
4,UG,2016,5,1,0.367388,32.594357,GPS,9999.0,1226.0,WGS84,44.857692,rich


### FB Reach

In [ ]:
### SET-UP
profiles = {'FBM_frequent_traveler':FacebookMarketing.BEHAVIOR_FREQUENT_TRAVELER,
            'FBM_small_business_owner':FacebookMarketing.BEHAVIOR_SMALL_BUSINESS_OWNER,
            'FBM_commuter':FacebookMarketing.BEHAVIOR_COMMUTER,
            'FBM_lives_abroad':FacebookMarketing.BEHAVIOR_LIVES_ABROAD,
            'FBM_frequent_int_traveler':FacebookMarketing.BEHAVIOR_FREQUENT_INTERNATIONAL_TRAVELER,
            'FBM_network_2G':FacebookMarketing.BEHAVIOR_NETWORK_2G,
            'FBM_network_3G':FacebookMarketing.BEHAVIOR_NETWORK_3G,
            'FBM_network_4G':FacebookMarketing.BEHAVIOR_NETWORK_4G,
            'FBM_feature_phone':FacebookMarketing.BEHAVIOR_FEATURE_PHONE,
            'FBM_old_device_os':FacebookMarketing.BEHAVIOR_OLD_DEVICE_OS,
            'FBM_mobile_access':FacebookMarketing.BEHAVIOR_MOBILE_ACCESS,
            'FBM_browser_access':FacebookMarketing.BEHAVIOR_BROWSER_ACCESS,
            'FBM_smartphone_tablet':FacebookMarketing.BEHAVIOR_SMARTPHONE_TABLET,
            'FBM_wifi':FacebookMarketing.BEHAVIOR_WIFI,
            'FBM_tech_early_adopter':FacebookMarketing.BEHAVIOR_TECHNOLOGY_EARLY_ADOPTERS,
            'FBM_returned_travel_1week':FacebookMarketing.BEHAVIOR_RETURNED_TRAVEL_1WEEK,
            'FBM_away_from_home':FacebookMarketing.LIFEEVENT_AWAY_FROM_HOME,
            'FBM_engaged':FacebookMarketing.LIFEEVENT_ENGAGED,
            'FBM_ind_gov':FacebookMarketing.INDUSTRY_GOV,
            'FBM_ind_business':FacebookMarketing.INDUSTRY_BUSINESS,
            'FBM_ind_legal':FacebookMarketing.INDUSTRY_LEGAL,
            'FBM_ind_it':FacebookMarketing.INDUSTRY_IT,
            'FBM_device_motorola':FacebookMarketing.DEVICE_MOTOROLA,
            'FBM_device_amazon':FacebookMarketing.DEVICE_AMAZON,
            'FBM_device_nokia':FacebookMarketing.DEVICE_NOKIA,
            'FBM_device_microsoft':FacebookMarketing.DEVICE_MICROSOFT,
            'FBM_os_ios':FacebookMarketing.OS_IOS,
            'FBM_os_android':FacebookMarketing.OS_ANDROID,
            'FBM_os_windows':FacebookMarketing.OS_WINDOWS,
            'FBM_os_windows_desktop':FacebookMarketing.OS_WINDOWS_DESKTOP,
            'FBM_os_windows_phone':FacebookMarketing.OS_WINDOWS_PHONE,
            'FBM_highschool':FacebookMarketing.EDU_HIGHSCHOOL,
            'FBM_bachelor':FacebookMarketing.EDU_BACHELOR,
            'FBM_master':FacebookMarketing.EDU_MASTER,
            'FBM_professional':FacebookMarketing.EDU_PROFESSIONAL,
            'FBM_phd':FacebookMarketing.EDU_PHD,
            'FBM_casino':FacebookMarketing.INTEREST_CASINO}

radius = geo.MILE_TO_M / geo.KM_TO_M
unit = FacebookMarketing.UNIT_KM
CACHE_DIR_LOCAL = '/content/FBM' 
CACHE_DIR_DRIVE = '/mydrive/data/Uganda/cache/FBM'
SLEEP = 60 * 30
MAXTOKENS = 10
print(radius, unit)

1.60934 kilometer


In [ ]:
!mkdir -p  $CACHE_DIR_LOCAL
!rsync $CACHE_DIR_DRIVE/ $CACHE_DIR_LOCAL

In [ ]:
!ls $CACHE_DIR_LOCAL | wc -lc
!ls $CACHE_DIR_DRIVE | wc -lc

  34014 2399647
  34014 2399647


In [ ]:
def fbconnect(tokenid):
  ### TOKENS
  fn_tokens = '/mydrive/data/Facebook/MarketingAPI/tokens{}.json'.format(tokenid)
  tokens = ios.load_json(fn_tokens)
  access_token = tokens['access_token']
  app_secret = tokens['app_secret']
  ad_account_id = tokens['ad_account_id']
  app_id = tokens['app_id']

  ### CONNECT
  fbm = FacebookMarketing(app_id=app_id,
                          app_secret=app_secret,
                          access_token=access_token,
                          ad_account_id=ad_account_id)
  fbm.init()
  return fbm

In [ ]:
from collections import defaultdict
from itertools import cycle
import time
import numpy as np
from datetime import datetime
import os

df_dhs_new = df_dhs.copy()
df_dhs_new = df_dhs_new[['LONGNUM','LATNUM']]

tokenids = cycle(np.arange(1,MAXTOKENS+1,1))
status = defaultdict(int)

tokenid = next(tokenids)
fbm = fbconnect(tokenid)
counter = 0
for id, row in df_dhs_new.iterrows():
  counter += 1
  print("========== {} of {} ==========".format(counter, df_dhs_new.shape[0]))
  
  ### QUERY
  for k,v in profiles.items():
    code = None
    while code not in [-1,0,2]:
      result, code = fbm.get_reach_estimate(lon=row.LONGNUM, lat=row.LATNUM, radius=radius, unit=unit, specific_params=v, cache_dir=CACHE_DIR_LOCAL) 
      value = None if result is None else result['users']
      df_dhs_new.loc[id,k] = value
      status[tokenid] = code == 1

      if code not in [-1,0]:
        print('code:', code)
      
      if status[tokenid]:
        print("token:{}, id:{}, result:fail".format(tokenid, id))
        
        if sum(list(status.values())) == MAXTOKENS:
          status = {k:0 for k,v in status.items()}
          current_time = datetime.now().strftime("%H:%M:%S")
          print('{}: sleeping {}min...'.format(current_time, SLEEP/60))
          
          ### save
          fn = '/mydrive/data/Uganda/results/DHS2016_MIS2018_iwi_cluster_Marketing.csv'
          ios.save_csv(df_dhs_new.drop(columns=['LONGNUM','LATNUM'], inplace=False), fn , index=True)
          ### cache to drive
          os.system("rsync -r {}/ {}".format(CACHE_DIR_LOCAL, CACHE_DIR_DRIVE))
          ### sleep 
          time.sleep(SLEEP)

        tokenid = next(tokenids)
        fbm = fbconnect(tokenid)

    if code==2:
      # skip wrong location
      break

os.system("rsync -r {}/ {}".format(CACHE_DIR_LOCAL, CACHE_DIR_DRIVE))

========== 1 of 1001 ==========
========== 2 of 1001 ==========
========== 3 of 1001 ==========
========== 4 of 1001 ==========
========== 5 of 1001 ==========
========== 6 of 1001 ==========
========== 7 of 1001 ==========
========== 8 of 1001 ==========
========== 9 of 1001 ==========
========== 10 of 1001 ==========
========== 11 of 1001 ==========
========== 12 of 1001 ==========
========== 13 of 1001 ==========
========== 14 of 1001 ==========
========== 15 of 1001 ==========
========== 16 of 1001 ==========
========== 17 of 1001 ==========
========== 18 of 1001 ==========
========== 19 of 1001 ==========
========== 20 of 1001 ==========
========== 21 of 1001 ==========
========== 22 of 1001 ==========
========== 23 of 1001 ==========
========== 24 of 1001 ==========
========== 25 of 1001 ==========
========== 26 of 1001 ==========
========== 27 of 1001 ==========
========== 28 of 1001 ==========
========== 29 of 1001 ==========
========== 30 of 1001 ==========
========== 31 of 10

0

In [ ]:
df_dhs_new.drop(columns=['LONGNUM','LATNUM'], inplace=True)
df_dhs_new.head()

,FBM_frequent_traveler,FBM_small_business_owner,FBM_commuter,FBM_lives_abroad,FBM_frequent_int_traveler,FBM_network_2G,FBM_network_3G,FBM_network_4G,FBM_feature_phone,FBM_old_device_os,FBM_mobile_access,FBM_browser_access,FBM_smartphone_tablet,FBM_wifi,FBM_tech_early_adopter,FBM_away_from_home,FBM_ind_gov,FBM_ind_business,FBM_ind_legal,FBM_ind_it,FBM_device_motorola,FBM_device_amazon,FBM_device_nokia,FBM_device_microsoft,FBM_os_ios,FBM_os_android,FBM_os_windows,FBM_os_windows_desktop,FBM_os_windows_phone,FBM_highschool,FBM_bachelor,FBM_master,FBM_professional,FBM_phd
0,66000.0,7400.0,1000.0,6600.0,66000.0,1000.0,37000.0,15000.0,1000.0,19000.0,30000.0,8700.0,63000.0,6100.0,2900.0,4400.0,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0,2800.0,1000.0,6800.0,110000.0,1500.0,1500.0,1000.0,14000.0,3700.0,40000.0,1000.0,1000.0
1,68000.0,7400.0,1000.0,6800.0,68000.0,1000.0,38000.0,15000.0,1000.0,19000.0,31000.0,9000.0,65000.0,6100.0,2900.0,4700.0,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0,2700.0,1000.0,7200.0,110000.0,1400.0,1400.0,1000.0,14000.0,3600.0,40000.0,1000.0,1000.0
2,75000.0,8200.0,1000.0,7300.0,74000.0,1000.0,41000.0,17000.0,1000.0,21000.0,33000.0,9700.0,71000.0,6900.0,3300.0,5000.0,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0,3100.0,1000.0,7600.0,120000.0,1600.0,1600.0,1000.0,16000.0,4000.0,45000.0,1000.0,1000.0
3,42000.0,4500.0,1000.0,4100.0,42000.0,1000.0,23000.0,9600.0,1000.0,12000.0,19000.0,5500.0,40000.0,3800.0,1800.0,2900.0,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0,1700.0,1000.0,4500.0,67000.0,1000.0,1000.0,1000.0,8800.0,2300.0,25000.0,1000.0,1000.0
4,580000.0,72000.0,31000.0,68000.0,580000.0,5600.0,270000.0,170000.0,6700.0,180000.0,290000.0,88000.0,580000.0,75000.0,31000.0,55000.0,1000.0,5500.0,1000.0,7000.0,3100.0,1000.0,26000.0,1000.0,54000.0,890000.0,16000.0,16000.0,1000.0,120000.0,44000.0,390000.0,1000.0,1000.0
